# 📌 Laboratorio: Algoritmos de Búsqueda y Factor de Aglomeración
## ✏️ PARTE 1: Datos del Grupo
---
- **Universidad:** Universidad César Vallejo (UCV)
- **Escuela Profesional:** Ingeniería de Sistemas
- **Ciclo:** VIII Ciclo
- **Grupo:** **G03**
- **Integrantes:**
  1. **Cordova Espinoza, Andy Jhoan** ([ORCID: 0009-0008-7854-8806](https://orcid.org/0009-0008-7854-8806)) - acordovaes2@ucvvirtual.edu.pe
  2. **Paredes Montoya, Raul Alexander** ([ORCID: 0009-0005-1728-1675](https://orcid.org/0009-0005-1728-1675)) - raparedesmo@ucvvirtual.edu.pe
  3. **Ramirez Bardales, Rober Kener** ([ORCID: 0009-0005-0065-6290](https://orcid.org/0009-0005-0065-6290)) - roramirezba@ucvvirtual.edu.pe
  4. **Rosales Villafuerte, Daniel Guillermo** ([ORCID: 0009-0005-6577-1943](https://orcid.org/0009-0005-6577-1943)) - drosalesvi@ucvvirtual.edu.pe


## ✏️ PARTE 2: Programación de Algoritmos de Búsqueda
En esta sección se implementan las 4 funciones requeridas:
1. `dfs()` — Búsqueda en Profundidad con Pila (LIFO)
2. `bfs()` — Búsqueda en Amplitud con Cola (FIFO)
3. `heuristica_manhattan()` — Distancia Manhattan $|x_1 - x_2| + |y_1 - y_2|$
4. `astar()` — Algoritmo A* con Cola de Prioridad (`heapq`) y costo $f(n) = g(n) + h(n)$


In [ ]:
import heapq
from collections import deque
import time
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------
# 1. HEURÍSTICA MANHATTAN
# --------------------------------------------------------------------------
def heuristica_manhattan(nodo_actual, objetivo):
    """
    Calcula la distancia Manhattan entre dos puntos 2D: |x1 - x2| + |y1 - y2|.
    Admisible y consistente para movimientos ortogonales en cuadrícula.
    """
    x1, y1 = nodo_actual
    x2, y2 = objetivo
    return abs(x1 - x2) + abs(y1 - y2)


# --------------------------------------------------------------------------
# 2. BÚSQUEDA EN PROFUNDIDAD (DFS) - Usando una PILA (LIFO)
# --------------------------------------------------------------------------
def dfs(grafo, inicio, objetivo):
    """
    Búsqueda en Profundidad (DFS) utilizando una Pila (Stack / LIFO).
    Retorna: (camino, nodos_explorados)
    """
    pila = [inicio]
    padres = {inicio: None}
    visitados = set()
    nodos_explorados = 0
    
    while pila:
        actual = pila.pop()  # Comportamiento LIFO: extrae el último agregado
        nodos_explorados += 1
        
        if actual == objetivo:
            break
            
        if actual in visitados:
            continue
        visitados.add(actual)
        
        # Explorar vecinos
        for vecino in grafo.get(actual, []):
            if vecino not in visitados:
                if vecino not in padres:
                    padres[vecino] = actual
                pila.append(vecino)  # Se apila para continuar en profundidad
                
    if objetivo not in padres:
        return None, nodos_explorados
        
    # Reconstrucción del camino desde el objetivo hasta el inicio
    camino = []
    curr = objetivo
    while curr is not None:
        camino.append(curr)
        curr = padres[curr]
    camino.reverse()
    return camino, nodos_explorados


# --------------------------------------------------------------------------
# 3. BÚSQUEDA EN AMPLITUD (BFS) - Usando una COLA (FIFO)
# --------------------------------------------------------------------------
def bfs(grafo, inicio, objetivo):
    """
    Búsqueda en Amplitud (BFS) utilizando una Cola (Queue / FIFO).
    Garantiza el camino más corto en grafos no ponderados.
    Retorna: (camino, nodos_explorados)
    """
    cola = deque([inicio])
    padres = {inicio: None}
    visitados = {inicio}
    nodos_explorados = 0
    
    while cola:
        actual = cola.popleft()  # Comportamiento FIFO: extrae el primero en llegar
        nodos_explorados += 1
        
        if actual == objetivo:
            break
            
        for vecino in grafo.get(actual, []):
            if vecino not in visitados:
                visitados.add(vecino)
                padres[vecino] = actual
                cola.append(vecino)  # Se encola para explorar por niveles
                
    if objetivo not in padres:
        return None, nodos_explorados
        
    # Reconstrucción del camino
    camino = []
    curr = objetivo
    while curr is not None:
        camino.append(curr)
        curr = padres[curr]
    camino.reverse()
    return camino, nodos_explorados


# --------------------------------------------------------------------------
# 4. BÚSQUEDA A* (A-STAR) - Usando heapq (Cola de Prioridad)
# --------------------------------------------------------------------------
def astar(grafo, inicio, objetivo, pesos=None):
    """
    Búsqueda A* guiada por la heurística Manhattan y evaluada con f(n) = g(n) + h(n).
    Utiliza un montículo binario (heapq) para extraer eficientemente el nodo de menor f(n).
    Retorna: (camino, nodos_explorados, costo_total)
    """
    cola_prioridad = []
    contador = 0  # Desempate para elementos con idéntico f_score
    
    # Inicializar con g(inicio) = 0 y f(inicio) = h(inicio)
    h_inicio = heuristica_manhattan(inicio, objetivo)
    heapq.heappush(cola_prioridad, (h_inicio, contador, inicio))
    
    padres = {inicio: None}
    g_cost = {inicio: 0}
    visitados = set()
    nodos_explorados = 0
    
    while cola_prioridad:
        f_actual, _, actual = heapq.heappop(cola_prioridad)
        nodos_explorados += 1
        
        if actual == objetivo:
            break
            
        if actual in visitados:
            continue
        visitados.add(actual)
        
        for vecino in grafo.get(actual, []):
            # Costo de la arista (1 por defecto en cuadrícula estándar)
            costo_arista = 1 if pesos is None else pesos.get((actual, vecino), 1)
            nuevo_g = g_cost[actual] + costo_arista
            
            if nuevo_g < g_cost.get(vecino, float('inf')):
                g_cost[vecino] = nuevo_g
                padres[vecino] = actual
                f_score = nuevo_g + heuristica_manhattan(vecino, objetivo)
                contador += 1
                heapq.heappush(cola_prioridad, (f_score, contador, vecino))
                
    if objetivo not in padres:
        return None, nodos_explorados, float('inf')
        
    # Reconstrucción del camino óptimo
    camino = []
    curr = objetivo
    while curr is not None:
        camino.append(curr)
        curr = padres[curr]
    camino.reverse()
    return camino, nodos_explorados, g_cost.get(objetivo, 0)

print("✅ Funciones de Parte 2 cargadas exitosamente.")


## ✏️ PARTE 3: Generación del Gráfico Propio y Análisis Comparativo
Se ejecuta una prueba sobre un entorno de navegación de cuadrícula con obstáculos y se comparan métricas clave de desempeño:
- **Nodos Explorados** (Carga computacional)
- **Longitud / Costo del Camino** (Optimalidad de la ruta)
- **Tiempo de Ejecución** (ms)


In [ ]:
# --------------------------------------------------------------------------
# CREACIÓN DEL ENTORNO DE PRUEBA (Cuadrícula 15x15 con obstáculos)
# --------------------------------------------------------------------------
filas, columnas = 15, 15
obstaculos = set([
    (3, c) for c in range(2, 12)
] + [
    (8, c) for c in range(3, 14)
] + [
    (r, 7) for r in range(4, 8)
])

# Construir grafo de adyacencia de la cuadrícula
grafo_cuadricula = {}
movimientos = [(-1, 0), (1, 0), (0, -1), (0, 1)]
for r in range(filas):
    for c in range(columnas):
        if (r, c) in obstaculos:
            continue
        vecinos = []
        for dr, dc in movimientos:
            nr, nc = r + dr, c + dc
            if 0 <= nr < filas and 0 <= nc < columnas and (nr, nc) not in obstaculos:
                vecinos.append((nr, nc))
        grafo_cuadricula[(r, c)] = vecinos

inicio = (0, 0)
meta = (14, 14)

# --------------------------------------------------------------------------
# EJECUCIÓN Y MEDICIÓN DE MÉTRICAS
# --------------------------------------------------------------------------
# 1. DFS
t0 = time.perf_counter()
camino_dfs, exp_dfs = dfs(grafo_cuadricula, inicio, meta)
t_dfs = (time.perf_counter() - t0) * 1000
costo_dfs = len(camino_dfs) - 1 if camino_dfs else 0

# 2. BFS
t0 = time.perf_counter()
camino_bfs, exp_bfs = bfs(grafo_cuadricula, inicio, meta)
t_bfs = (time.perf_counter() - t0) * 1000
costo_bfs = len(camino_bfs) - 1 if camino_bfs else 0

# 3. A*
t0 = time.perf_counter()
camino_ast, exp_ast, costo_ast = astar(grafo_cuadricula, inicio, meta)
t_ast = (time.perf_counter() - t0) * 1000

print(f"DFS -> Pasos: {costo_dfs}, Nodos explorados: {exp_dfs}, Tiempo: {t_dfs:.3f} ms")
print(f"BFS -> Pasos: {costo_bfs}, Nodos explorados: {exp_bfs}, Tiempo: {t_bfs:.3f} ms")
print(f"A*  -> Pasos: {costo_ast}, Nodos explorados: {exp_ast}, Tiempo: {t_ast:.3f} ms")

# --------------------------------------------------------------------------
# RENDERIZADO DEL GRÁFICO COMPARATIVO
# --------------------------------------------------------------------------
algoritmos = ["DFS (Pila)", "BFS (Cola)", "A* (heapq + Manhattan)"]
nodos_explorados = [exp_dfs, exp_bfs, exp_ast]
longitudes_camino = [costo_dfs, costo_bfs, costo_ast]
tiempos_ms = [t_dfs, t_bfs, t_ast]
colores = ["#E74C3C", "#3498DB", "#2ECC71"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gráfica 1: Nodos Explorados
bars1 = axes[0].bar(algoritmos, nodos_explorados, color=colores, alpha=0.9, edgecolor="black")
axes[0].set_title("Nodos Explorados (Menos es mejor)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Cantidad de nodos expandidos")
axes[0].grid(axis="y", linestyle="--", alpha=0.6)
for bar in bars1:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, yval + 1, f"{int(yval)}", ha="center", va="bottom", fontweight="bold")

# Gráfica 2: Longitud del Camino (Costo)
bars2 = axes[1].bar(algoritmos, longitudes_camino, color=colores, alpha=0.9, edgecolor="black")
axes[1].set_title("Longitud del Camino (Optimalidad)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Pasos / Distancia total")
axes[1].grid(axis="y", linestyle="--", alpha=0.6)
for bar in bars2:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, yval + 0.5, f"{int(yval)}", ha="center", va="bottom", fontweight="bold")

# Gráfica 3: Tiempo de Ejecución
bars3 = axes[2].bar(algoritmos, tiempos_ms, color=colores, alpha=0.9, edgecolor="black")
axes[2].set_title("Tiempo de Ejecución (ms)", fontsize=12, fontweight="bold")
axes[2].set_ylabel("Milisegundos")
axes[2].grid(axis="y", linestyle="--", alpha=0.6)
for bar in bars3:
    yval = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.2f} ms", ha="center", va="bottom", fontweight="bold")

plt.suptitle("Laboratorio Grupo G03 - Comparativa de Algoritmos de Búsqueda", fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()


### 📝 Interpretación Académica de Resultados (Parte 3)

1. **Optimalidad de Ruta (BFS vs A* vs DFS):**
   - **BFS (Búsqueda en Amplitud)** y **A*** garantizan encontrar el camino más corto (óptimo) hacia la meta en un grafo uniforme. En este experimento ambos hallan la ruta de menor costo exacto.
   - **DFS (Búsqueda en Profundidad)** no es un algoritmo óptimo. Debido a la naturaleza LIFO de la pila, explora un ramal hasta el límite sin considerar la distancia acumulada, lo que resulta en un camino serpenteante y significativamente más largo que el óptimo.

2. **Eficiencia y Espacio de Búsqueda (Nodos Explorados):**
   - **A*** demuestra una eficiencia superior al expandir sustancialmente menos nodos que BFS. Esto se debe al uso de la **heurística Manhattan**, la cual actúa como una guía informada hacia el objetivo, priorizando los nodos que reducen la distancia estimada y podando ramas improductivas.
   - **BFS**, al ser una búsqueda no informada (ciega), se expande uniformemente en círculos concéntricos (radio radial), explorando nodos innecesarios en direcciones opuestas a la meta antes de encontrar el destino.

3. **Estructuras de Datos y Complejidad:**
   - **Pila (DFS):** Requiere poco espacio de memoria $O(b \cdot m)$, pero es propensa a quedar atrapada en caminos largos y no óptimos.
   - **Cola (BFS):** Requiere almacenar toda la frontera en memoria $O(b^d)$, lo que satura la memoria rápidamente conforme crece el radio de búsqueda.
   - **Cola de Prioridad (`heapq` - A*):** Equilibra de forma óptima el costo real $g(n)$ y el costo estimado $h(n)$, obteniendo el mejor balance entre precisión de ruta y velocidad computacional.
